In [1]:
import os
from typing import TypedDict, List, Literal, Optional
from dotenv import load_dotenv

load_dotenv()

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.callbacks import BaseCallbackHandler
from langchain_community.vectorstores import FAISS
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel, Field

llm = ChatOpenAI(model="gpt-4o-mini")
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [2]:
# adaptive rag : 쿼리 -> 검색을 할지 말지
# self rag : retrieved 문서 

In [3]:
class SelfRAGState(TypedDict):
    question : str
    rewritten_question: str  # C-RAG 
    documents : List[Document]
    filtered_documents : List[Document]
    generation : str
    relevance_grades : List[str]
    hallucination_score : str
    usefulness_score : int
    relevance_retries : int
    generation_retries : int
    max_tries : int

In [4]:
builder = StateGraph(SelfRAGState)
builder.add_edge(START, END)
app = builder.compile()

In [5]:
DOCS = [
    Document(page_content="Self-RAG는 검색·생성·평가를 LLM이 스스로 판단하는 자가 평가 RAG입니다.",
             metadata={"src": "selfrag-intro"}),
    Document(page_content="원논문 Self-RAG는 reflection token을 학습시켜 검색 여부와 답변 품질을 표시합니다.",
             metadata={"src": "selfrag-paper"}),
    Document(page_content="구현체는 LLM-as-judge 패턴(with_structured_output)으로 reflection token을 대신합니다.",
             metadata={"src": "selfrag-impl"}),
    Document(page_content="관련성 평가에서 yes 문서만 컨텍스트로 추리고, 모두 no면 쿼리 재작성 또는 재검색.",
             metadata={"src": "selfrag-relevance"}),
    Document(page_content="환각 평가는 답변이 문서에 근거하는지 확인합니다. yes(환각)면 다시 생성합니다.",
             metadata={"src": "selfrag-hallucination"}),
    Document(page_content="유용성 평가는 답변이 질문에 정말 답하는지 1-5점으로 매깁니다. 3점 미만이면 재생성.",
             metadata={"src": "selfrag-usefulness"}),
    Document(page_content="LangGraph는 StateGraph로 노드와 조건부 엣지를 조립해 평가 루프를 자연스럽게 표현합니다.",
             metadata={"src": "langgraph"}),
    Document(page_content="쿼리 재작성은 검색 실패 시 동의어/구체화로 쿼리를 바꿔 다시 검색하는 폴백 패턴입니다.",
             metadata={"src": "query-rewrite"}),
]
vectorstore = FAISS.from_documents(DOCS, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [6]:
def retrieve_node(state : SelfRAGState) -> dict:
    docs = retriever.invoke(state['question'])
    return {'documents' : docs}

In [7]:
INIT = {"question": "", "rewritten_question": "",
        "documents": [], "filtered_documents": [], "generation": "",
        "relevance_grades": [], "hallucination_score": "",
        "usefulness_score": 0,
        "relevance_retries": 0, "generation_retries": 0, "max_retries": 4}

In [8]:
builder = StateGraph(SelfRAGState)
builder.add_node('retrieve', retrieve_node)
builder.add_edge(START, 'retrieve')
builder.add_edge('retrieve', END)
app = builder.compile()

In [9]:
result = app.invoke({**INIT, "question" : 'self-rag의 환각 평가는 어떻게 하나요?'})
result

{'question': 'self-rag의 환각 평가는 어떻게 하나요?',
 'rewritten_question': '',
 'documents': [Document(id='1fc8516b-90e1-4b98-828a-a6486386c830', metadata={'src': 'selfrag-intro'}, page_content='Self-RAG는 검색·생성·평가를 LLM이 스스로 판단하는 자가 평가 RAG입니다.'),
  Document(id='1fd043ea-74e1-42c2-b49c-7f853d98c124', metadata={'src': 'selfrag-paper'}, page_content='원논문 Self-RAG는 reflection token을 학습시켜 검색 여부와 답변 품질을 표시합니다.'),
  Document(id='581fb752-929a-4a0b-bd63-9caf13850d5a', metadata={'src': 'selfrag-hallucination'}, page_content='환각 평가는 답변이 문서에 근거하는지 확인합니다. yes(환각)면 다시 생성합니다.')],
 'filtered_documents': [],
 'generation': '',
 'relevance_grades': [],
 'hallucination_score': '',
 'usefulness_score': 0,
 'relevance_retries': 0,
 'generation_retries': 0}

In [10]:
RAG_PROMPT = ChatPromptTemplate.from_messages([
    ('system', "다음 문서를 근거로 한국어로 간단히 답하세요. 문서에 없으면 '모름'이라고 답하세요"),
    ('human', "질문 : {question}\n\n문서:\n{context}")
])

def generate_node(state : SelfRAGState) -> dict:
    ctx = '\n'.join(f" - {d.page_content}" for d in state['documents'])
    msg = RAG_PROMPT.format_messages(question = state['question'], context = ctx)
    return {'generation' : llm.invoke(msg).content}

In [11]:
builder = StateGraph(SelfRAGState)
builder.add_node('retrieve', retrieve_node)
builder.add_node('generate', generate_node)
builder.add_edge(START, 'retrieve')
builder.add_edge('retrieve', 'generate')
builder.add_edge('generate', END)
app = builder.compile()

In [12]:
result = app.invoke({**INIT, 'question' : 'self-rag의 유용성 평가 기준은?'})
result

{'question': 'self-rag의 유용성 평가 기준은?',
 'rewritten_question': '',
 'documents': [Document(id='1fc8516b-90e1-4b98-828a-a6486386c830', metadata={'src': 'selfrag-intro'}, page_content='Self-RAG는 검색·생성·평가를 LLM이 스스로 판단하는 자가 평가 RAG입니다.'),
  Document(id='1fd043ea-74e1-42c2-b49c-7f853d98c124', metadata={'src': 'selfrag-paper'}, page_content='원논문 Self-RAG는 reflection token을 학습시켜 검색 여부와 답변 품질을 표시합니다.'),
  Document(id='6bbf749b-08c9-4775-971d-f589ce998d05', metadata={'src': 'selfrag-usefulness'}, page_content='유용성 평가는 답변이 질문에 정말 답하는지 1-5점으로 매깁니다. 3점 미만이면 재생성.')],
 'filtered_documents': [],
 'generation': 'self-RAG의 유용성 평가 기준은 답변이 질문에 정말 답하는지 1-5점으로 평가하는 것입니다. 3점 미만이면 재생성합니다.',
 'relevance_grades': [],
 'hallucination_score': '',
 'usefulness_score': 0,
 'relevance_retries': 0,
 'generation_retries': 0}

In [13]:
# retrieve -> grade_relevance -> generate

In [14]:
class _Relevance(BaseModel):
    binary : Literal['yes', 'no'] = Field(description='문서가 질문과 관련이 있는지')

relevance_grader = llm.with_structured_output(_Relevance)

REL_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "당신은 문서가 질문과 관련 있는지 평가하는 전문가입니다. yes/no 로만 반환해주세요"),
    ("human", "질문: {question}\n문서: {doc}")
])

def grade_relevance_node(state : SelfRAGState) -> dict:
    # llm documents를 평가해서 relevant 여부를 yes/no 답변
    grades = []
    for d in state['documents']:
        msg = REL_PROMPT.format_messages(question = state['question'], doc = d.page_content) # yes, no, yes입니다, no. , 
        grades.append(relevance_grader.invoke(msg).binary)
    return {'relevance_grades' : grades}

In [15]:
builder = StateGraph(SelfRAGState)
builder.add_node('retrieve', retrieve_node)
builder.add_node('grade_relevance', grade_relevance_node)
builder.add_node('generate', generate_node)

builder.add_edge(START, 'retrieve')
builder.add_edge('retrieve', 'grade_relevance')
builder.add_edge('grade_relevance', 'generate')
builder.add_edge('generate', END)
app = builder.compile()

In [16]:
result = app.invoke({**INIT, 'question' : 'self-rag의 유용성 평가 기준은?'})

/home/oncreative/anaconda3/envs/modu/lib/python3.11/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=_Relevance(binary='yes'), input_type=_Relevance])
  return self.__pydantic_serializer__.to_python(
/home/oncreative/anaconda3/envs/modu/lib/python3.11/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=_Relevance(binary='no'), input_type=_Relevance])
  return self.__pydantic_serializer__.to_python(


In [17]:
result

{'question': 'self-rag의 유용성 평가 기준은?',
 'rewritten_question': '',
 'documents': [Document(id='1fc8516b-90e1-4b98-828a-a6486386c830', metadata={'src': 'selfrag-intro'}, page_content='Self-RAG는 검색·생성·평가를 LLM이 스스로 판단하는 자가 평가 RAG입니다.'),
  Document(id='1fd043ea-74e1-42c2-b49c-7f853d98c124', metadata={'src': 'selfrag-paper'}, page_content='원논문 Self-RAG는 reflection token을 학습시켜 검색 여부와 답변 품질을 표시합니다.'),
  Document(id='6bbf749b-08c9-4775-971d-f589ce998d05', metadata={'src': 'selfrag-usefulness'}, page_content='유용성 평가는 답변이 질문에 정말 답하는지 1-5점으로 매깁니다. 3점 미만이면 재생성.')],
 'filtered_documents': [],
 'generation': 'self-rag의 유용성 평가 기준은 답변이 질문에 정말 답하는지를 1-5점으로 매기는 것입니다. 3점 미만이면 재생성합니다.',
 'relevance_grades': ['yes', 'yes', 'no'],
 'hallucination_score': '',
 'usefulness_score': 0,
 'relevance_retries': 0,
 'generation_retries': 0}

In [18]:
result = app.invoke({**INIT, 'question' : '제로콜라 가격은?'})

In [19]:
result

{'question': '제로콜라 가격은?',
 'rewritten_question': '',
 'documents': [Document(id='1fc8516b-90e1-4b98-828a-a6486386c830', metadata={'src': 'selfrag-intro'}, page_content='Self-RAG는 검색·생성·평가를 LLM이 스스로 판단하는 자가 평가 RAG입니다.'),
  Document(id='22b8e6dd-5b41-4228-be5d-10628162fbfa', metadata={'src': 'selfrag-relevance'}, page_content='관련성 평가에서 yes 문서만 컨텍스트로 추리고, 모두 no면 쿼리 재작성 또는 재검색.'),
  Document(id='2404cfaf-f7c4-4a71-a95b-026d987e5542', metadata={'src': 'query-rewrite'}, page_content='쿼리 재작성은 검색 실패 시 동의어/구체화로 쿼리를 바꿔 다시 검색하는 폴백 패턴입니다.')],
 'filtered_documents': [],
 'generation': '모름',
 'relevance_grades': ['no', 'no', 'no'],
 'hallucination_score': '',
 'usefulness_score': 0,
 'relevance_retries': 0,
 'generation_retries': 0}

In [ ]:
# retrieve -> grade_relevance -> router  -관련된 문서가 있음-> generate
      ^                            |
      ------------관련문서없음------|


In [ ]:
def grade_relevance_node(state : SelfRAGState) -> dict:
    # llm documents를 평가해서 relevant 여부를 yes/no 답변
    grades = []
    for d in state['documents']:
        msg = REL_PROMPT.format_messages(question = state['question'], doc = d.page_content) # yes, no, yes입니다, no. , 
        grades.append(relevance_grader.invoke(msg).binary)
    return {'relevance_grades' : grades}

In [20]:
def grade_and_filter_node(state : SelfRAGState) -> dict:
    grades, keep = [], []
    for d in state['documents']:
        msg = REL_PROMPT.format_messages(question = state['question'], doc = d.page_content)
        g = relevance_grader.invoke(msg).binary
        grades.append(g)
        if g == 'yes':
            keep.append(d)
    return {'relevance_grades' : grades, 'filtered_documents' : keep}

def generate_filtered_node(state : SelfRAGState) -> dict:
    ctx = '\n'.join(f" - {d.page_content}" for d in state['filtered_documents'])
    msg = RAG_PROMPT.format_messages(question = state['question'], context = ctx)
    return {'generation' : llm.invoke(msg).content}

def relevance_router(state : SelfRAGState) -> dict:
    return "generate" if state['filtered_documents'] else "retrieve"

In [ ]:
# def grade_and_filter_with_counter_node(state : SelfRAGState) -> dict:
#     grades, keep = [], []
#     for d in state['documents']:
#         msg = REL_PROMPT.format_messages(question = state['question'], doc = d.page_content)
#         g = relevance_grader.invoke(msg).binary
#         grades.append(g)
#         if g == 'yes':
#             keep.append(d)
#     return {'relevance_grades' : grades, 'filtered_documents' : keep, 'relevance_retries' : state['relevance_retries'] + 1}  

# def safe_relevance_router(state : SelfRAGState) -> dict:
#     if state['relevance_retries'] >=2:
#         return 'generate'
#     return "generate" if state['filtered_documents'] else "retrieve"

In [21]:
builder = StateGraph(SelfRAGState)
builder.add_node('retrieve', retrieve_node)
builder.add_node('grade_relevance', grade_and_filter_node)
builder.add_node('generate', generate_filtered_node)

builder.add_edge(START, 'retrieve')
builder.add_edge('retrieve', 'grade_relevance')
builder.add_conditional_edges('grade_relevance', relevance_router, {'generate' : 'generate', 'retrieve' : 'retrieve'})
builder.add_edge('generate', END)
app = builder.compile()

In [22]:
result = app.invoke({**INIT, 'question' : 'self-rag의 관련성 평가는?'})

In [23]:
result

{'question': 'self-rag의 관련성 평가는?',
 'rewritten_question': '',
 'documents': [Document(id='1fc8516b-90e1-4b98-828a-a6486386c830', metadata={'src': 'selfrag-intro'}, page_content='Self-RAG는 검색·생성·평가를 LLM이 스스로 판단하는 자가 평가 RAG입니다.'),
  Document(id='1fd043ea-74e1-42c2-b49c-7f853d98c124', metadata={'src': 'selfrag-paper'}, page_content='원논문 Self-RAG는 reflection token을 학습시켜 검색 여부와 답변 품질을 표시합니다.'),
  Document(id='6bbf749b-08c9-4775-971d-f589ce998d05', metadata={'src': 'selfrag-usefulness'}, page_content='유용성 평가는 답변이 질문에 정말 답하는지 1-5점으로 매깁니다. 3점 미만이면 재생성.')],
 'filtered_documents': [Document(id='1fc8516b-90e1-4b98-828a-a6486386c830', metadata={'src': 'selfrag-intro'}, page_content='Self-RAG는 검색·생성·평가를 LLM이 스스로 판단하는 자가 평가 RAG입니다.'),
  Document(id='1fd043ea-74e1-42c2-b49c-7f853d98c124', metadata={'src': 'selfrag-paper'}, page_content='원논문 Self-RAG는 reflection token을 학습시켜 검색 여부와 답변 품질을 표시합니다.')],
 'generation': 'Self-RAG의 관련성 평가는 LLM이 스스로 판단하여 검색 여부와 답변 품질을 표시하는 방식으로 이루어집니다.',
 'relevance_grades': [

In [25]:
# result = app.invoke({**INIT, 'question' : '제로콜라의 가격은?'})

In [ ]:
# retrieve -> grade_relevance -> router -> generate -> END
#  ^                               |
# ----------rewrite query----------

In [ ]:
class SelfRAGState(TypedDict):
    question : str
    rewritten_question: str  # C-RAG 
    documents : List[Document]
    filtered_documents : List[Document]
    generation : str
    relevance_grades : List[str]
    hallucination_score : str
    usefulness_score : int
    relevance_retries : int
    generation_retries : int
    max_tries : int

In [26]:
REWRITE_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "다음 질문을 의미는 보존하되 검색이 더 잘 되도록 동의어, 구체화로 재작성해줘. 한 줄로만."),
    ("human", "원래 질문: {question}")
])

def rewrite_query_node(state : SelfRAGState) -> dict :
    msg = REWRITE_PROMPT.format_messages(question = state['question'])
    new_q = llm.invoke(msg).content.strip()
    return {'rewritten_question' : new_q} # 제로콜라 얼마지?

def retrieve_aware_node(state : SelfRAGState) -> dict :
    q = state['rewritten_question'] or state['question']
    return {'documents' : retriever.invoke(q)}

def relevance_router(state : SelfRAGState) -> dict:
    if state['filtered_documents']:
        return 'generate'
    return 'rewrite_query'
    

In [27]:
builder = StateGraph(SelfRAGState)
builder.add_node('retrieve', retrieve_aware_node)
builder.add_node('grade_relevance', grade_and_filter_node)
builder.add_node('rewrite_query', rewrite_query_node)
builder.add_node('generate', generate_filtered_node)

builder.add_edge(START, 'retrieve')
builder.add_edge('retrieve', 'grade_relevance')
builder.add_conditional_edges('grade_relevance', relevance_router, {'generate' : 'generate', 'rewrite_query' : 'rewrite_query'})
builder.add_edge('rewrite_query', 'retrieve')
builder.add_edge('generate', END)
app = builder.compile()

In [35]:
# result = app.invoke({**INIT, 'question' : '제로콜라 가격은?'})

In [ ]:
# rewrite_query : 쿼리 재작성 횟수는 최대 1회

In [36]:
def rewrite_with_counter(state : SelfRAGState) -> dict:
    msg = REWRITE_PROMPT.format_messages(question = state['question'])
    return {'rewritten_question' : llm.invoke(msg).content.strip(), 'relevance_retries' : state['relevance_retries'] +1}
def bounded_router(state : SelfRAGState) -> dict:
    if state['filtered_documents']:
        return 'generate'
    if state['relevance_retries'] <1:
        return 'rewrite_query'
    return 'generate'

In [ ]:
# retrieve -> grade_relevance -> router -> generate -> grade_hallucination -> END
#  ^                               |
# ----------rewrite query----------

In [41]:
class _Hallucination(BaseModel):
    binary : Literal['yes', 'no'] = Field(description='답변이 환각인지 (yes = 문서 밖 정보, no = 근거 있음)')

hallucination_grader = llm.with_structured_output(_Hallucination)

HAL_PROMT = ChatPromptTemplate.from_messages([
    ("system", "당신은 엄격한 채점관입니다. 답변의 모든 사실이 문서에 명시적으로 있어야 no. 일반 지식, 문서 밖 정보가 있으면 yes(환각), 근거하면 no"),
    ("human", "문서: \n{context}\n\n답변: {answer}")
])

def grade_hallucination_node(state : SelfRAGState) -> dict:
    docs = state['filtered_documents'] or state['docments']
    ctx = '\n'.join(d.page_content for d in docs)
    msg = HAL_PROMT.format_messages(context = ctx, answer = state['generation'])
    return {'hallucination_score' : hallucination_grader.invoke(msg).binary}

In [38]:
builder = StateGraph(SelfRAGState)
builder.add_node('retrieve', retrieve_aware_node)
builder.add_node('grade_relevance', grade_and_filter_node)
builder.add_node('rewrite_query', rewrite_query_node)
builder.add_node('generate', generate_filtered_node)
builder.add_node('grade_hallucination', grade_hallucination_node)

builder.add_edge(START, 'retrieve')
builder.add_edge('retrieve', 'grade_relevance')
builder.add_conditional_edges('grade_relevance', relevance_router, {'generate' : 'generate', 'rewrite_query' : 'rewrite_query'})
builder.add_edge('rewrite_query', 'retrieve')
builder.add_edge('generate', 'grade_hallucination')
builder.add_edge('grade_hallucination', END)
app = builder.compile()

In [39]:
result = app.invoke({**INIT, 'question' : 'self-rag 환각 평가는 어떻게 하나요?'})

/home/oncreative/anaconda3/envs/modu/lib/python3.11/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=_Hallucination(binary='no'), input_type=_Hallucination])
  return self.__pydantic_serializer__.to_python(


In [40]:
result

{'question': 'self-rag 환각 평가는 어떻게 하나요?',
 'rewritten_question': '',
 'documents': [Document(id='1fc8516b-90e1-4b98-828a-a6486386c830', metadata={'src': 'selfrag-intro'}, page_content='Self-RAG는 검색·생성·평가를 LLM이 스스로 판단하는 자가 평가 RAG입니다.'),
  Document(id='581fb752-929a-4a0b-bd63-9caf13850d5a', metadata={'src': 'selfrag-hallucination'}, page_content='환각 평가는 답변이 문서에 근거하는지 확인합니다. yes(환각)면 다시 생성합니다.'),
  Document(id='1fd043ea-74e1-42c2-b49c-7f853d98c124', metadata={'src': 'selfrag-paper'}, page_content='원논문 Self-RAG는 reflection token을 학습시켜 검색 여부와 답변 품질을 표시합니다.')],
 'filtered_documents': [Document(id='1fc8516b-90e1-4b98-828a-a6486386c830', metadata={'src': 'selfrag-intro'}, page_content='Self-RAG는 검색·생성·평가를 LLM이 스스로 판단하는 자가 평가 RAG입니다.')],
 'generation': '모름',
 'relevance_grades': ['yes', 'no', 'no'],
 'hallucination_score': 'no',
 'usefulness_score': 0,
 'relevance_retries': 0,
 'generation_retries': 0}